In [1]:
!pip install -q sentence_transformers
! pip install -q torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 2.8 MB/s eta 0:00:00


In [2]:
from typing import List
from sentence_transformers import SentenceTransformer, util, CrossEncoder
import torch

if not torch.cuda.is_available():
    print("Warning: No GPU found. Please add GPU to your notebook")

# We use the Bi-Encoder to encode all passages, so that we can use it with semantic search
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
bi_encoder.max_seq_length = 256     # Truncate long passages to 256 tokens
top_k = 32                          # Number of passages we want to retrieve with the bi-encoder

# The bi-encoder will retrieve 100 documents. We use a cross-encoder, to re-rank the results list to improve the quality
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def search2(query, passages, top_k=3):
    # Find potentially relevant passages using Bi-Encoder
    query_embedding = bi_encoder.encode(query)
    corpus_embeddings = bi_encoder.encode(passages)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)

    # Extract the hits from the result
    hits = hits[0]

    top_k_texts_bi_encoder = [passages[hit['corpus_id']] for hit in hits]

    # Use Cross-Encoder to re-rank the top-k texts from Bi-Encoder
    input_pairs = [(query, text) for text in top_k_texts_bi_encoder]
    scores = cross_encoder.predict(input_pairs)
    sorted_indices_cross_encoder = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    top_k_text_chunks_cross_encoder = [top_k_texts_bi_encoder[i] for i in sorted_indices_cross_encoder[:top_k]]

    # Create a sorted table of scores along with the text chunks
    scores_table = [(scores[i], top_k_texts_bi_encoder[i]) for i in sorted_indices_cross_encoder]

    return top_k_text_chunks_cross_encoder, scores_table


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [3]:
# Define the predefined labels
standardized_sentences = [
    "actifs",
    "caisse_et_avoirs",
    "creances_sur_etablissements_bancaires_et_financiers",
    "creances_sur_la_clientele",
    "portefeuille_titres_commercial",
    "portefeuille_investissement",
    "valeurs_immobilisees",
    "autres_actifs",
    "total_actifs",
    "passifs",
    "banque_centrale_et_ccp",
    "depots_et_avoirs_etablissements_bancaires_et_financiers",
    "depots_et_avoirs_clientele",
    "emprunts_et_ressources_speciales",
    "depot_abg",
    "autres_ressources",
    "total_emprunts_ressources_speciales",
    "autres_passifs",
    "total_passifs",
    "capitaux_propres",
    "capital",
    "reserves",
    "resultats_reportes",
    "resultat_de_la_periode",
    "total_capitaux_propres",
    "total_passifs_et_capitaux_propres"
]

# Sample DataFrame
data = {
    "BILAN TRIMESTRIEL ARRETE AU 31 MARS 2018": [
        "AC 1 - Portefeuille-titres  4.1.1 5 055 325 7 549 765  5 241 939",
        "a - Actions, valeurs assimilées et droits rattachés   300 822 231 363  300 274",
        "b - Obligations et valeurs assimilées   4 754 503 7 318 402  4 941 665",
        "c - Autres valeurs",
        "AC 2 - Placements monétaires et disponibilités  4.1.2 1 077 844 453 180  1 139 999",
        "a - Placements monétaires   403 392   301 208",
        "b - Disponibilités   674 452   838 791",
        "AC 3 - Créances d'exploitation  4.1.3     90",
        "a - Dividendes et intérets a recevoir",
        "b - titres de créance échus     -",
        "AC 4 - Autres actifs  4.1.4 20",
        "a - Débiteurs divers",
        "b - Immobilisations nettes",
        "TOTAL ACTIF  6 133 189 8 002 965  6 382 048",
        "PASSIE",
        "PA 1 - Dettes sur opérations de pensions livrées  4.1.5  897 259",
        "PA 2 - Opérateurs créditeurs  4.1.6 10 896 5 495  9 839",
        "PA 3 - Autres créditeurs divers  4.1.7 18 143 13 155  13 119",
        "TOTAL PASSIF  29 039 915 909  22 958",
        "ACTIF NET",
        "CP 1 - Capital  4.1.8 5 873 516 6 761 648  6 171 739",
        "CP 2 - Sommes distribuables   230 634 325 408  187 351",
        "a - Sommes distribuables des exercices antérieurs   178 283 257 769   41",
        "b - Sommes distribuables de la période   52 351 67 639  187 310",
        "TOTAL PASSIF ET ACTIF NET"
    ]
}

In [4]:
import re

def process_dataframe(data):
    """
    Process the DataFrame to extract relevant information.

    Args:
    data (dict): Dictionary containing column names as keys and lists of data as values.

    Returns:
    dict: Dictionary containing extracted text as values and extracted numbers as keys.
    """
    extracted_text = []
    extracted_numbers = []
    for col_data in data.values():
        for line in col_data:
            # Remove numbers and punctuation, keep only alphabetical characters
            clean_text = re.sub(r'[^a-zA-Z\s]', '', line)
            extracted_numbers.append(clean_text)

            # Find all numeric values in the line
            numbers = re.findall(r'\b\d[\d\s.]+\b', line)
            extracted_text.extend(numbers)
    # Create a dictionary with extracted numbers as keys and extracted text as values
    extracted_data = dict(zip(extracted_numbers, extracted_text))
    return extracted_data


In [5]:
financial_data = process_dataframe(data)

In [6]:
def standardize_financial_data(standardized_sentences, financial_data):
    # Initialize a list to store the standardized financial data
    standardized_list = []

    # Iterate over each element in the standardized sentences
    for sentence in standardized_sentences:
        # Perform semantic search to find the most relevant financial data
        top_k_text_chunks_cross_encoder, scores_table = search2(sentence, list(financial_data.keys()), top_k=5)

        # If no matching financial data found, append None
        if not top_k_text_chunks_cross_encoder:
            standardized_list.append((sentence, None))
            continue

        # Select the financial data with the highest score
        best_match = top_k_text_chunks_cross_encoder[0]
        best_score = None

        # Iterate through the scores table to handle redundancy
        for score, matched_data in scores_table:
            if matched_data == best_match:
                best_score = score
                break

        # If redundancy exists, select the next best match from the scores table
        if best_score is not None:
            for score, matched_data in scores_table:
                if score < best_score and matched_data not in [data for _, data in standardized_list]:
                    best_match = matched_data
                    break

        # Add the selected financial data to the standardized list
        standardized_list.append((sentence, financial_data[best_match]))

    return standardized_list


In [7]:
standarized_list = standardize_financial_data(standardized_sentences, financial_data)
print(standarized_list)

[('actifs', '4.1.8 5 873 516 6 761 648  6 171 739'), ('caisse_et_avoirs', '4.1.2 1 077 844 453 180  1 139 999'), ('creances_sur_etablissements_bancaires_et_financiers', '4.1.1 5 055 325 7 549 765  5 241 939'), ('creances_sur_la_clientele', '4.1.4 20'), ('portefeuille_titres_commercial', '4.1.5  897 259'), ('portefeuille_investissement', '4.1.2 1 077 844 453 180  1 139 999'), ('valeurs_immobilisees', '4.1.2 1 077 844 453 180  1 139 999'), ('autres_actifs', '4.1.2 1 077 844 453 180  1 139 999'), ('total_actifs', '4.1.6 10 896 5 495  9 839'), ('passifs', '4.1.8 5 873 516 6 761 648  6 171 739'), ('banque_centrale_et_ccp', '4.1.1 5 055 325 7 549 765  5 241 939'), ('depots_et_avoirs_etablissements_bancaires_et_financiers', '4.1.1 5 055 325 7 549 765  5 241 939'), ('depots_et_avoirs_clientele', '4.1.1 5 055 325 7 549 765  5 241 939'), ('emprunts_et_ressources_speciales', '4.1.8 5 873 516 6 761 648  6 171 739'), ('depot_abg', '4.1.1 5 055 325 7 549 765  5 241 939'), ('autres_ressources', '4.1.

In [9]:
print("standarized_sentences lenght",len(standardized_sentences), "standarized_list lenght ", len(standarized_list))

standarized_sentences lenght 26 standarized_list lenght  26


In [11]:
import json
json_results = json.dumps(standarized_list, indent=2)
print(json_results)

[
  [
    "actifs",
    "4.1.8 5 873 516 6 761 648  6 171 739"
  ],
  [
    "caisse_et_avoirs",
    "4.1.2 1 077 844 453 180  1 139 999"
  ],
  [
    "creances_sur_etablissements_bancaires_et_financiers",
    "4.1.1 5 055 325 7 549 765  5 241 939"
  ],
  [
    "creances_sur_la_clientele",
    "4.1.4 20"
  ],
  [
    "portefeuille_titres_commercial",
    "4.1.5  897 259"
  ],
  [
    "portefeuille_investissement",
    "4.1.2 1 077 844 453 180  1 139 999"
  ],
  [
    "valeurs_immobilisees",
    "4.1.2 1 077 844 453 180  1 139 999"
  ],
  [
    "autres_actifs",
    "4.1.2 1 077 844 453 180  1 139 999"
  ],
  [
    "total_actifs",
    "4.1.6 10 896 5 495  9 839"
  ],
  [
    "passifs",
    "4.1.8 5 873 516 6 761 648  6 171 739"
  ],
  [
    "banque_centrale_et_ccp",
    "4.1.1 5 055 325 7 549 765  5 241 939"
  ],
  [
    "depots_et_avoirs_etablissements_bancaires_et_financiers",
    "4.1.1 5 055 325 7 549 765  5 241 939"
  ],
  [
    "depots_et_avoirs_clientele",
    "4.1.1 5 055 325 7 54

In [ ]:
# Define the predefined labels for état de résultat
standardized_sentencesET = [
    "Charges d'exploitation",
    "Variation de stock de produits finis",
    "Achats d'approvisionnements consommés",
    "Charges du personnel",
    "Dotations aux amortissements et aux provisions",
    "Autres charges d'exploitation",
    "Total des charges d'exploitation",
    "Résultat d'exploitation",
    "Produits financiers nets",
    "Produits de placements",
    "Autres gains ordinaires",
    "Autres pertes ordinaires",
    "Résultat des activités ordinaires avant impôt",
    "Impôt sur les bénéfices",
    "Résultat des activités ordinaires après impôt",
    "Eléments extraordinaires (gains/pertes)",
    "Résultat net de l'exercice",
    "Effet des modifications comptables",
    "Résultat après modifications comptables"
]



In [ ]:

    # Define the predefined labels for état des flux de trésorerie
standardized_sentencesFT = [
    "Flux de trésorerie liés à l'exploitation",
    "Encaissements reçus des clients",
    "Sommes versées aux fournisseurs et au personnel",
    "Intérêts payés",
    "Impôts et taxes payés à l'état",
    "Autres flux d'exploitation",
    "Total des flux de trésorerie liés à l'exploitation",
    "Flux de trésorerie liés aux activités d'investissement",
    "Décaissements affectés à l'acquisition d'immobilisations corporelles et incorporelles",
    "Encaissements provenant de la cession d'immobilisations corporelles et incorporelles",
    "Décaissements affectés à l'acquisition d'immobilisations financières",
    "Encaissement de subventions",
    "Total des flux de trésorerie liés aux activités d'investissement",
    "Flux de trésorerie liés aux activités de financement",
    "Dividendes et autres distributions",
    "Encaissement provenant des mobilisations des créances",
    "Remboursement d'emprunts en principal",
    "Variation des mobilisations des créances",
    "Total des flux de trésorerie liés aux activités de financement",
    "Incidence des variations des taux de change sur les liquidités et équivalents de liquidités",
    "Variation de trésorerie",
    "Trésorerie au début de l'exercice",
    "Trésorerie à la clôture"
]
